In [0]:
# ============================================================
# Gold Layer - Delta Live Tables (DLT) Pipeline
# Spotify Analytics - Curated / Gold Layer
# Reads from Silver DLT tables in the same pipeline
# ============================================================

import dlt
from pyspark.sql import functions as F

print("Gold DLT pipeline loaded")

In [0]:
# ============================================================
# CELL 2: Gold - Top Tracks by Streams
# ============================================================

@dlt.table(
    name="gold_top_tracks",
    comment="Top tracks ranked by total stream count",
    table_properties={"quality": "gold"}
)
def gold_top_tracks():
    streams = spark.read.table("spotify_catalog.silver.fact_streams")
    tracks = spark.read.table("spotify_catalog.silver.dim_tracks")
    artists = spark.read.table("spotify_catalog.silver.dim_artists")

    return (
        streams
        .join(tracks, streams.track_id == tracks.track_id, "left")
        .join(artists, tracks.artist_id == artists.artist_id, "left")
        .groupBy(tracks.track_id, tracks.track_name, artists.artist_name, tracks.album_name)
        .agg(F.count("*").alias("total_streams"))
        .orderBy(F.desc("total_streams"))
        .limit(100)
    )

In [0]:
# ============================================================
# CELL 3: Gold - Top Artists by Streams
# ============================================================

@dlt.table(
    name="gold_top_artists",
    comment="Top artists ranked by total stream count",
    table_properties={"quality": "gold"}
)
def gold_top_artists():
    streams = spark.read.table("spotify_catalog.silver.fact_streams")
    tracks = spark.read.table("spotify_catalog.silver.dim_tracks")
    artists = spark.read.table("spotify_catalog.silver.dim_artists")

    return (
        streams
        .join(tracks, streams.track_id == tracks.track_id, "left")
        .join(artists, tracks.artist_id == artists.artist_id, "left")
        .groupBy(artists.artist_id, artists.artist_name)
        .agg(F.count("*").alias("total_streams"))
        .orderBy(F.desc("total_streams"))
        .limit(50)
    )

In [0]:
# ============================================================
# CELL 4: Gold - Streams by Date
# DLT Materialized View - Daily stream volume
# ============================================================

@dlt.table(
    name="gold_streams_by_date",
    comment="Daily stream counts for trend analysis",
    table_properties={"quality": "gold"}
)
def gold_streams_by_date():
    streams = spark.read.table("spotify_catalog.silver.fact_streams")
    dates = spark.read.table("spotify_catalog.silver.dim_date")

    return (
        streams
        .join(dates, streams.date_key == dates.date_key, "left")
        .groupBy(dates.date_key, dates.date, dates.year, dates.month, dates.day)
        .agg(F.count("*").alias("total_streams"))
        .orderBy(dates.year, dates.month, dates.day)
    )

In [0]:
# ============================================================
# CELL 5: Gold - Monthly Streams Trend
# DLT Materialized View - Monthly aggregated stream counts
# ============================================================

@dlt.table(
    name="gold_monthly_streams",
    comment="Monthly stream counts aggregated for trend analysis",
    table_properties={"quality": "gold"}
)
def gold_monthly_streams():
    streams = spark.read.table("spotify_catalog.silver.fact_streams")
    dates = spark.read.table("spotify_catalog.silver.dim_date")

    return (
        streams
        .join(dates, streams.date_key == dates.date_key, "left")
        .groupBy(dates.year, dates.month)
        .agg(F.count("*").alias("total_streams"))
        .orderBy(dates.year, dates.month)
    )

In [0]:
# ============================================================
# CELL 6: Gold - Enriched User Listening History
# DLT Materialized View - Denormalized fact table
# Joins all dimensions for a wide gold table
# ============================================================

@dlt.table(
    name="gold_user_listening_history",
    comment="Enriched user listening history with track, artist, album, user, and date details",
    table_properties={"quality": "gold"}
)
def gold_user_listening_history():
    streams = spark.read.table("spotify_catalog.silver.fact_streams")
    tracks = spark.read.table("spotify_catalog.silver.dim_tracks")
    artists = spark.read.table("spotify_catalog.silver.dim_artists")
    users = spark.read.table("spotify_catalog.silver.dim_users")
    dates = spark.read.table("spotify_catalog.silver.dim_date")

    return (
        streams
        .join(tracks, streams.track_id == tracks.track_id, "left")
        .join(artists, tracks.artist_id == artists.artist_id, "left")
        .join(users, streams.user_id == users.user_id, "left")
        .join(dates, streams.date_key == dates.date_key, "left")
        .select(
            streams.stream_id,
            streams.user_id,
            users.user_name,
            streams.track_id,
            tracks.track_name,
            tracks.album_name,
            artists.artist_id,
            artists.artist_name,
            dates.date,
            dates.year,
            dates.month,
            streams.listen_duration,
            streams.device_type,
            streams.stream_timestamp
        )
    )